**Word-based tokenization:**
- Most used from th epast. Easy to set up and use with only a few rules and often yields decent results. 
- Many different ways to split the text for example, we could use whitespaces to tokenize the text into words by applying python's split function

In [1]:
tokenized_text = "Jim Henson was a puppeteer".split()
print(tokenized_text)

['Jim', 'Henson', 'was', 'a', 'puppeteer']


- different variations exist with different extra rules for punctuation. with this kind of tokenization, we usually end up with very large vocaabularies (vocabulary being defined as the total number of independent tokens in the corpus). Each word, gets assigned an ID starting from 0 and going up to the size of the vocabulary. The model then uses these IDs to identify each word. 
- To completely cover a language, we need to have an identifier for each word in the language, which generates a huge amount of tokens. Furthermore, words like dog and dogs are represented differently initially and the model will have no way of knowing they are similar initially. 
- Finally, we also risk having unknown tokens in the event that the test set has words that weren't in the training set.

**Character Based Tokenization:**
- Character-based tokenizers split the text into characters, rather than words and this has 2 primary benefits
  - The vocabulary is much smaller
  - There are much fewer out-of-vocabulary (unknown) tokens, since every word can be built from characters
- this approach isn't perfect either since the representation is now based on characters rather than words, one could argue that, intuitively, its less meaningful: each character doesn't mean much on its own whereas that is the case with words. However, this again differs according to the language, where in some languages characters carry more information
- Another thing to consider is that we'll end up with a very large number of tokens to be processed by the model: whereas a word would only be a single token with a word-based tokenizer, it can easily turn into 10 or more tokens when converted into characters.

- To get the best of both worlds, we can use a third technique that combines the two approaches: subword tokenization. 

**Subword tokenization:**
- need to find the balance between both techniques: character based tokenization and word based tokenization
- word based tokenization downsides: very large vocabularies, large quantities of out-of-vocab tokens, loss of meaning across very similar words
- character based tokenization downsides: very long sequences, less meaningful individual tokens
- 2 main principles of subword tokenization: frequently used words should not be split into subwords, while rarely used words should be decomposed into meaningful subwords
- Subwords end up providing a lot of semantic meaning: for instance, in the example above, tokenization was s
- Subword tokenization algos:
   - Byte-level BPE, as used in GPT-2
   - WordPiece, as used in BERT
   - SentencePiece or Unigram, as used in several multilingual models

High Level Stuff:
- using the tokenization api
- **Loading and Saving:**
  - loading and saving tokenizers is based on 2 methods: from_pretrained() to load and save_pretrained to save the model. These methods will load or save the algorithm used by the tokenizer (a bit like the architecture of the model), as well as its vocabulary
  - Loading the BERT tokenizer trained with the same checkpoint as BERT is done this way:

In [2]:
from transformers import BertTokenizer

In [3]:
tokenizer = BertTokenizer.from_pretrained("bert-base-cased")

Alternatively, can use the AutoTokenizer class which will grab the proper tokenizer class in the library based on the checkpoint name, and can be used directly with any checkpoint

In [4]:
from transformers import AutoTokenizer

In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

we can now ise the tokenizer.....

In [6]:
tokenizer("Using a Transformer network is simple")

{'input_ids': [101, 7993, 170, 13809, 23763, 2443, 1110, 3014, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

saving the tokenizer:
- tokenizer.save_pretrained("directory")

**Encoding:**
- Translating text to numbers: encoding. Encoding is done in a 2 step process: the tokenization, followed by the conversion to input ids
- First step: split the text into words - or tokens. second step is to convert the tokens to numbers. so we can build a tensor out of them and feed them to the model. to do this, the tokenizer has a vocabulary, which is the part we download when we instantiate it with the from_pretrained method. 
- **Tokenization:**
   - the tokenization process is done by the tokenize() method of the tokenizer:

In [7]:
from transformers import AutoTokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")
sequence = "Using a Transformer network is simple"
tokens = tokenizer.tokenize(sequence)

In [9]:
tokens

['Using', 'a', 'Trans', '##former', 'network', 'is', 'simple']

the tokenizer used is a subword tokenizer: it splits the words until it obtains tokens that can be represented by its vocabulary. That's the case here with transformer (so the above step is the tokenizer performing greedy tokenization) based on the vocab it was trained on

**From tokens to input IDs:**
- the conversion to input IDs is handled by the convert_tokens_to_ids() tokenizer method:

In [10]:
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids)

[7993, 170, 13809, 23763, 2443, 1110, 3014]


the above outputs, then once converted to the appropriate framework tensor, can then be used as inputs to a model

**Decoding:**
- Decoding is going the other way round: from vocabulary indices, we want to get a string. this can be done with the decode() method as follows:

In [11]:
decoded_string = tokenizer.decode([7993, 170, 13809, 23763, 2443, 1110, 3014])
print(decoded_string)

Using a Transformer network is simple


Abit more in depth:

**Build a tokenizer from scratch:**
- lets train a new tokenizer on wikitext-103

In [12]:
from datasets import load_dataset_builder

In [13]:
wk_text = load_dataset_builder("Salesforce/wikitext",'wikitext-103-raw-v1')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [14]:
wk_text.info.splits

{'test': SplitInfo(name='test', num_bytes=1305088, num_examples=4358, shard_lengths=None, dataset_name='wikitext'),
 'train': SplitInfo(name='train', num_bytes=546500949, num_examples=1801350, shard_lengths=[1648675, 152675], dataset_name='wikitext'),
 'validation': SplitInfo(name='validation', num_bytes=1159288, num_examples=3760, shard_lengths=None, dataset_name='wikitext')}

**Training the tokenizer**
- we build and train a Byte-Pair Encoding (BPE) tokenizer. 
- training the tokenizer means it will learn merge rules by:
  - start with all the characters present in the training corpus as tokens
  - identify the most common pair of tokens and merge into one token
  - repeat until the vocabulary (i.e. the number of tokens) has reached the size we want.

The main API of the library is the class Tokenizer, here's how we instantiate one with a BPE model.

In [15]:
from tokenizers import Tokenizer
from tokenizers.models import BPE

In [16]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

passing the unknown token to the bpe model because, the model needs to know what to do when it encounters something it cannot encode. This is runtime behavior not training. the model answers questions like im trying to encode text and cannot decompose this input into unknown subwords. what token id should i emit. 
- without the unknown token, the model literally doesn't know how to fail safely. 

To train our tokenizer on the wikitext files, we need to instantiate a trainer (trainer.title-ref), in this case a BpeTrainer....

In [17]:
from tokenizers.trainers import BpeTrainer

In [18]:
trainer = BpeTrainer(special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])

why bpe trainer needs special tokens:
- this on the other hand is vocabulary construction. which tokens must exist, no matter what. what token ids should be reserved. which tokens should never be split or merged. if unk is added, the trainer might never add it to the vocab, or could assign it a random id late in training or merges could accidentally involve it. 

we can set the training arguments like vocab_size or min_frequency (here left at their default values of 30,000 and 0) but the most important part is to give the special_tokens we plan to use later on (they're not using during training) so they get inserted into the vocabulary. (order of special tokens matters: unk=> id 0, cls=> id 1, etc....). 
- We could train our tokenizer now, but wouldn't be optimal. without a pre-tokenizer that will split our inputs into words, we might get tokens that overlap several words. BPE as an algo has no concept of words. It just sees a sequence of symbols and learns which adjacent symbols to merge. If you do not restrict it, BPE would merge across word boundaries. 
- so we have a pretokenizer that runs before BPE training and splits the raw text into chunks, and BPE isn't allowed to merge across those chunk boundaries
- whitespace pretokenization - splits the text into plain words.
- Using a pretokenizer ensures no token is bigger than a word returned by the pretokenizer. Here, we want to train a subword BPE tokenizer, and will use the easiest pre-tokenizer possible by splitting on whitespace.

In [19]:
from tokenizers.pre_tokenizers import Whitespace

In [20]:
tokenizer.pre_tokenizer = Whitespace()

Now, call the Tokenizer.train method with any list of files we want to use...

- 2 ways of training: tokenizer.train(files=....) takes a list of file paths (strings)
- tokenizer.train_from_iterator(...): an iterator of text strings (Iterable[str])

In [21]:
# load the dataset:
from datasets import load_dataset

data = load_dataset("Salesforce/wikitext",'wikitext-103-raw-v1')

In [22]:
data

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

In [23]:
def text_iterator(dataset_dict):
    for split in dataset_dict.keys():
        for i in range(dataset_dict[f'{split}'].num_rows):
            yield dataset_dict[f'{split}'][i]['text']

In [24]:
tokenizer.train_from_iterator(text_iterator(data), trainer=trainer)

to save the tokenizer in one file - just use the tokenizer.save method:
- tokenizer.save('path.json')

**Using the tokenizer:**
- Now that we have a trained tokenizer, we can use it on any text we want with the tokenizer.encode method:

In [25]:
output = tokenizer.encode("Hello, y'all! How are you 😁 ?")

In [26]:
output

Encoding(num_tokens=11, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

the tokens attribute of the returned object contains the segmentation of your text in tokens:

In [27]:
print(output.tokens)

['Hello', ',', 'y', "'", 'all', '!', 'How', 'are', 'you', '[UNK]', '?']


similarly, the ids attribute will contain the index of each of those tokens in the tokenizer's vocabulary

In [28]:
print(output.ids)

[27253, 16, 93, 11, 5097, 5, 7961, 5112, 6218, 0, 35]


an important feature of the hf tokenizers lib -> comes with full alignment tracking, meaning you can always get the part of your original sentence that corresponds to a given token. Those are stored in the offsets attribute of the encoding object. for instance, let's assume we would want to find back what caused the [UNK] token to appear. we can just for the offset at that index.

In [29]:
# unk appears at index 9 -> so here's how we find it
print(output.offsets[9]) # corresponding offsets

(26, 27)


In [30]:
sentence = "Hello, y'all! How are you 😁 ?"
sentence[26:27]

'😁'

**Post-processing:**
- might want our tokenizer to automatically add special tokens like 'CLS' or 'SEP'. to do this, we use a post-processor. TemplateProcessing is the most commonly used,  you just have to specify a template for the processing of single sentences and pairs of sentences, along withthe special tokens and their ids.
- when building the tokenizer, we set cls and sep in positions 1 and 2 of the special tokens so this should be their ids... to double check can use the tokenizer.token_to_id method:

In [31]:
tokenizer.token_to_id("[SEP]")

2

In [32]:
from tokenizers.processors import TemplateProcessing

first we specify the template for single sentences: these should be of form "[CLS] $A [SEP]" where $A reps the sentence. then the template for sentence pairs of form "[CLS] $A [SEP] $B [SEP]" where $A reps the first sentence and $B the second one. The :1 added in the template represent the type IDs we want for each part of our input: it defaults to 0 for everything (which is why we don't have $A:0), and here we set it to 1 for the tokens of the second sentence and the last '[SEP]' token. Lastly, we specify the special tokens we used and their IDs in our tokenizer's vocabulary. 

In [33]:
tokenizer.post_processor = TemplateProcessing(
    single = "[CLS] $A [SEP]",
    pair = "[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens = [
        ("[CLS]", tokenizer.token_to_id("[CLS]")),
        ("[SEP]", tokenizer.token_to_id("[SEP]"))
    ]
)

In [34]:
output = tokenizer.encode("Hello, y'all! How are you 😁 ?")
print(output.tokens)

['[CLS]', 'Hello', ',', 'y', "'", 'all', '!', 'How', 'are', 'you', '[UNK]', '?', '[SEP]']


To check the results on a pair of sentences, we just pass the two sentences to tokenizer.encode:

In [35]:
output = tokenizer.encode("Hello y'all","How are you 😁")

In [36]:
print(output.tokens)

['[CLS]', 'Hello', 'y', "'", 'all', '[SEP]', 'How', 'are', 'you', '[UNK]', '[SEP]']


you can then check the type IDs attributed to each token:

In [37]:
print(output.type_ids)

[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1]


saving your tokenizer with Tokenizer.save saves your post-processor as well...

**Encoding multiple sentences in a batch:**
- to get the full speed of the tokenizer lib, its best to process texts by batches using the Tokenizer.encode_batch mtd

In [38]:
output = tokenizer.encode_batch(["Hello, y'all!", "How are you 😁 ?"])

In [39]:
output

[Encoding(num_tokens=8, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing]),
 Encoding(num_tokens=7, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])]

the output is a list of encoding objects...like the ones we saw before. 
- can process together as many texts as long as they fit in memory. 
to process a batch of sentence pairs, pass two lists to the Tokenizer.encode_batch method: the list of sentences A and B and the list of sentences B:

In [40]:
output = tokenizer.encode_batch(
    [["Hello, y'all!", "How are you 😁 ?"], ["Hello to you too!", "I'm fine, thank you!"]]
)

when encoding multiple sentences, you can automatically pad the outputs to the longest sentence present by using Tokenizer.enable_padding, with the pad_token and its id 

In [41]:
tokenizer.token_to_id('[PAD]')

3

In [42]:
tokenizer.enable_padding(pad_id=3, pad_token='[PAD]')

We can set the direction of the padding (defaults to the right) or a given length or a given length if we want to pad every sample to that specific number (here we leave it unset to pad to the size of the longest text)

In [43]:
output = tokenizer.encode_batch(["Hello, y'all!", "How are you 😁 ?"])

In [44]:
print(output[1].tokens)

['[CLS]', 'How', 'are', 'you', '[UNK]', '?', '[SEP]', '[PAD]']


In [45]:
output[0].ids

[1, 27253, 16, 93, 11, 5097, 5, 2]

In [46]:
output[1].ids

[1, 7961, 5112, 6218, 0, 35, 2, 3]

In this case, the attention mask generated by the tokenizer takes the padding into account:

In [47]:
print(output[1].attention_mask)

[1, 1, 1, 1, 1, 1, 1, 0]


**Pretrained:**
- Any tokenizer can be loaded from the hf hub using:
  - AutoTokenizer
  - or Tokenizer class itself

In [48]:
from tokenizers import Tokenizer

In [49]:
tokenizer = Tokenizer.from_pretrained("bert-base-uncased")

**The tokenization pipeline:**
-  when calling Tokenizer.encode or Tokenizer.encode_batch, the input text(s) go through the following pipeline:
   - normalization
   - pre-tokenization
   - model
   - post-processing

**Normalization:**
- set of operations applied to a raw string to make it less random or cleaner. 
- Common operations include stripping whitespaces, removing accented characters or lowercasing all text
- Each normalization operation is represented in the hf tokenizers lib by a Normalizer, and you can combine several of those using a normalizers.Sequence. Here's a normalizer applying NFD and StripAccents

wtf is NFD:
- NFD = Normalization Form Decomposed
- Unicode standard -> takes characters that look like one symbol and splits them into base character + combining marks. 
  - Example: NFD("é")  →  "e◌́"
- Then followed by stripaccents => StripAccents removes combining marks
   - applying it after NFD deletes the accent part
   - NFD -> StripAccents
     - NFD("é") → "e◌́"
     - StripAccents("e◌́") → "e"
   - prevents the vocabulary from exploding, better frequency statistics, better generalization

In [50]:
from tokenizers import normalizers
from tokenizers.normalizers import NFD, StripAccents

In [51]:
normalizer = normalizers.Sequence([NFD(), StripAccents()])

In [52]:
# can manually test that normalizer by applying it to any string:
normalizer.normalize_str("Héllò hôw are ü?")

'Hello how are u?'

when building a tokenizer can customize its normalizer by just changing the correspoding attribute:

In [53]:
tokenizer.normalizer

BertNormalizer(clean_text=True, handle_chinese_chars=True, strip_accents=None, lowercase=True)

In [54]:
tokenizer.normalizer = normalizer

of course after changing the way a normalizer applies normalization you should probably retrain it from scratch afterwards..

checking out other normalizers out there just for curiosity's sake....

In [55]:
bert_normalizer = normalizers.BertNormalizer()

In [56]:
bert_normalizer.normalize_str("Héllò hôw are ü?")

'hello how are u?'

bert normalizer is the default normalizer attached to a tokenizer...

**Pre-Tokenization:**
- Pre-tokenization is the act of splitting a text into a smaller objects that give an upper bound to what your tokens will be at the end of training. A good way to think of this is that the pre-tokenizer will split your text into words and then, your final tokena will be paerts of those words
- An easy way to pre-tokenize inputs is to split on spaces and punctuations, which is done by the pre_tokenizers.Whitespace pre-tokenizer...

In [57]:
from tokenizers.pre_tokenizers import Whitespace

In [58]:
pre_tokenizer = Whitespace()

In [59]:
pre_tokenizer.pre_tokenize_str("Hello! How are you? I'm fine, thank you.")

[('Hello', (0, 5)),
 ('!', (5, 6)),
 ('How', (7, 10)),
 ('are', (11, 14)),
 ('you', (15, 18)),
 ('?', (18, 19)),
 ('I', (20, 21)),
 ("'", (21, 22)),
 ('m', (22, 23)),
 ('fine', (24, 28)),
 (',', (28, 29)),
 ('thank', (30, 35)),
 ('you', (36, 39)),
 ('.', (39, 40))]

Output is a list of tuples, with each tuple containing one word and its span in the original sentence. note: splitting on punctuation splits on contractions like 'I'm' in this example. Can combine any PreTokenizer together, for instance, here is a pre-tokenizer that will split on space, punctuation, and digits, separating numbers into their individual digits.....

In [60]:
from tokenizers import pre_tokenizers
from tokenizers.pre_tokenizers import Digits

In [61]:
pre_tokenizer = pre_tokenizers.Sequence([Whitespace(), Digits(individual_digits=True)])
pre_tokenizer.pre_tokenize_str("Call 911!")

[('Call', (0, 4)), ('9', (5, 6)), ('1', (6, 7)), ('1', (7, 8)), ('!', (8, 9))]

You can customize the pre-tokenizer of a tokenizer by changing the corresponding attribute..

In [62]:
tokenizer.pre_tokenizer = pre_tokenizer

Checking out other pre_tokenizers provided by the library...

In [63]:
# bert pre_tokenizer: splits tokens on spaces, and also on punctuation. Each occurrence of a punctuation character
# is treated separately
bert_ptokenizer = pre_tokenizers.BertPreTokenizer()

In [64]:
bert_ptokenizer.pre_tokenize_str("Call 911")

[('Call', (0, 4)), ('911', (5, 8))]

In [65]:
bert_ptokenizer.pre_tokenize_str("Hello! How are you? I'm fine, thank you.")

[('Hello', (0, 5)),
 ('!', (5, 6)),
 ('How', (7, 10)),
 ('are', (11, 14)),
 ('you', (15, 18)),
 ('?', (18, 19)),
 ('I', (20, 21)),
 ("'", (21, 22)),
 ('m', (22, 23)),
 ('fine', (24, 28)),
 (',', (28, 29)),
 ('thank', (30, 35)),
 ('you', (36, 39)),
 ('.', (39, 40))]

In [66]:
ws_pretokenizer = pre_tokenizers.Whitespace()

In [67]:
ws_pretokenizer.pre_tokenize_str("Hello, world!")

[('Hello', (0, 5)), (',', (5, 6)), ('world', (7, 12)), ('!', (12, 13))]

**ByteLevel pre_tokenizer:**
- this pre-tokenizer takes care of replacing all bytes of the given string with a correspinding representation as well as splitting into words.
- returns: a list of characters that compose the alphabet
- returns the alphabet used by this pretokenizer
- since the bytelevel works as its name suggests, at the byte level, it encides each byte into a unique visible character. This means that there is a total of 256 different characters composing this alphabet.

In [68]:
byte_pretokenizer = pre_tokenizers.ByteLevel()

In [69]:
byte_pretokenizer.pre_tokenize_str("Hello, World!")

[('ĠHello', (0, 5)), (',', (5, 6)), ('ĠWorld', (6, 12)), ('!', (12, 13))]

ByteLevel pre-tokenizer:
- maps each byte to a printable unicode character (using the gpt-2 byte mapping)
- marks whitespaces using a special symbol
GPT-2 Byte -> Unicode mapping:
- each byte is mapped to a visible unicode character.
  - space (byte 32) -> Ġ
  - letters remain readable (utf-8 is ascii compatible)
- then comes the pre-tokenization (splitting):
  - the byte level pre-tokenizer:
    - splits on byte-encoded whitespace
    - keeps punctuations as separate tokens
- pretokenizer exposes the byte boundaries and BPE learns which byte sequences to merge.

In [73]:
byte_pretokenizer.pre_tokenize_str("Hello, there 😊 😁 😡")

[('ĠHello', (0, 5)),
 (',', (5, 6)),
 ('Ġthere', (6, 12)),
 ('ĠðŁĺĬ', (12, 14)),
 ('ĠðŁĺģ', (14, 16)),
 ('ĠðŁĺ¡', (16, 18))]

the emoji is not one byte. 😊 -> F0 9F 98 8A  (4 bytes). ByteLevel pretokenization maps each byte to a unicode character, but doesn't split tokens at every byte. 

**Model:**
- Once the input texts are normalized and pre-tokenized, the tokenizer applies the model on the pre-tokens. This is the part of the pipeline that needs training on the corpus (or that has been trained if using a pretrained tokenizer). Model's role -> split your words (tokens) using the rules it learned. Its also responsible for mapping those tokens to their corresponding IDs in the vocabulary of the model. 
- This model is passed along when initializing the tokenizer so you already know how to customize this paer.
- The tokenizers lib supports:
  - models.BPE
  - models.Unigram
  - models.WordLevel
  - models.WordPiece

- WordPiece (BERT's tokenizer):
  - Core idea: start with characters, keep merging if it improves likelihood, but only in a greedy left-to-right way at encoding time
    - word piece is merge-bases during training and greedy during inference  

**Postprocessing:**
- post-processing is the last step of the tokenization pipeline, to perform any additional transformations to the encoding before its returned, like adding potential special tokens.
- we can customize the post processor of a tokenizer by setting the corresponding attribute. for example - here's how we can post-process to make the inputs suitable for a bert model...

In [74]:
from tokenizers.processors import TemplateProcessing

In [75]:
tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]",
    pair = "[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]",1),("[SEP]",2)]
)

**All together: a BERT tokenizer from scratch:**
- putting all pieces together to build a BERT tokenizer. first, bert relies on wordpiece, so we instantiate a new tokenizer....

In [76]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece

In [77]:
bert_tokenizer = Tokenizer(WordPiece(unk_token="[UNK]"))

then, we know that bert preprocesses texts by removing accents and lowercasing. we also use a unicode normalizer:

In [78]:
from tokenizers import normalizers
from tokenizers.normalizers import NFD, Lowercase, StripAccents

In [79]:
bert_tokenizer.normalizer = normalizers.Sequence([NFD(), Lowercase(), StripAccents()])

the pre-tokenizer is just splitting on whitespace and punctuation

In [80]:
from tokenizers.pre_tokenizers import Whitespace

In [81]:
bert_tokenizer.pre_tokenizer = Whitespace()

the post-processing uses the template seen in the previous section:

In [82]:
from tokenizers.processors import TemplateProcessing

In [83]:
bert_tokenizer.post_processor = TemplateProcessing(
    single = "[CLS] $A [SEP]",
    pair = "[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[
        ("[CLS]",1),
        ("[SEP]",2)
    ]
)

we can now use this tokenizer and then train it on wikitext

In [84]:
from tokenizers.trainers import WordPieceTrainer

In [85]:
trainer = WordPieceTrainer(vocab_size=30522, special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]", "[MASK]"])
bert_tokenizer.train_from_iterator(text_iterator(data), trainer=trainer)

**Decoding:**
- encode: text to ids
- decode: ids back text
The decoder first converts the ids to tokens (using the vocabulary), removes special tokens and then joins these tokens with spaces

In [89]:
output = bert_tokenizer.encode("Hello, y'all! How are you 😁")
print(output.ids)

[1, 27462, 16, 67, 11, 7323, 5, 7510, 7268, 7989, 0, 2]


In [90]:
print(output.tokens)

['[CLS]', 'hello', ',', 'y', "'", 'all', '!', 'how', 'are', 'you', '[UNK]', '[SEP]']


In [91]:
bert_tokenizer.decode([1, 27462, 16, 67, 11, 7323, 5, 7510, 7268, 7989, 0, 2])

"hello , y ' all ! how are you"